In [40]:
import os
from typing import Any, Dict, List, TypedDict
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
# git=

USER_DATABASE = {
    "user123": {
        "name": "Siva Kumar", 
        "account_type": "Premium",
        "balance": 5000,
        "email": "siva@example.com",
        "support_tier": "Priority"
    },
    "user456": {
        "name": "Rishik Sharma",
        "account_type": "Standard", 
        "balance": 1200,
        "email": "rishik@example.com",
        "support_tier": "Standard"
    }
}

In [42]:
class UserContext(TypedDict):
    user_id: str

In [43]:
@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context["user_id"]
    
    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"""Account Information:
        Name: {user['name']}
        Type: {user['account_type']}  
        Balance: ${user['balance']}
        Email: {user['email']}
        Support Tier: {user['support_tier']}"""
    return "User not found in system"

@tool
def update_support_tier(new_tier: str, runtime: ToolRuntime[UserContext]) -> str:
    """Update user's support tier (Priority, Standard, Basic)."""
    user_id = runtime.context["user_id"]
    
    if user_id in USER_DATABASE:
        USER_DATABASE[user_id]['support_tier'] = new_tier
        return f"Support tier updated to {new_tier} for user {USER_DATABASE[user_id]['name']}"
    return "User not found"

In [44]:
agent = create_agent(
    model,
    tools=[get_account_info, update_support_tier],
    context_schema = UserContext,
    system_prompt = "You are a customer support assistant. Help users with their account inquirie"
)

In [45]:
# Test with real agent - Context is automatically injected into tools
print("=== Testing Context Access ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current account information?"}]},
    context=UserContext(user_id="user123")
)

=== Testing Context Access ===


In [46]:
print("\nAgent Response:")
print(result["messages"][-1].content)


Agent Response:
Your current account information is as follows:
- Name: Siva Kumar
- Type: Premium
- Balance: $5000
- Email: siva@example.com
- Support Tier: Priority

Is there anything specific you would like to do with your account?


LONG TERM MEMORY


In [47]:
# Namespace structure (tuple of segments)
namespace = ("category", "subcategory", "user_id")

# Examples:
("preferences", "user_prefs")
("notes", "personal")
("tasks", "user123", "workspace_alpha")
("app", "memory", "user123")   # per-user LTM bucket (used in this lab)

('app', 'memory', 'user123')

In [48]:
# Write — InMemoryStore.put requires a dict (string keys, JSON values), not a bare list.
store.put(namespace, key, {"messages": ["fact one", "fact two"]})

# Read — returns an Item (or None). item.value is that dict.
item = store.get(namespace, key)
if item is None:
    messages = []
else:
    messages = list(item.value.get("messages", []))

NameError: name 'store' is not defined

In [ ]:
class UserContext(TypedDict):
    user_id: str


def memory_namespace(user_id: str) -> tuple[str, ...]:
    """One LTM bucket per user — use this in every save/retrieve tool."""
    return ("app", "memory", user_id)

In [ ]:
LTM_KEY = "messages"


@tool
def save_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save a line of information to this user's long-term memory."""
    store = runtime.store
    ns = memory_namespace(runtime.context["user_id"])

    user_id = runtime.context["user_id"]
    item = store.get(ns, LTM_KEY)
    if item is None:
        messages: List[str] = []
    else:
        messages = list(item.value.get("messages", []))

    messages.append(data)
    print(f"adding to ltm before for {user_id}", data, messages)
    # InMemoryStore.put requires a dict (string keys, JSON values), not a list.
    store.put(ns, LTM_KEY, {"messages": messages})
    print("adding to ltm", data)
    return "Data Stored in long term memory"


@tool
def retrieve_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Retrieve this user's long-term memory (data arg can hint what to look for)."""
    store = runtime.store
    ns = memory_namespace(runtime.context["user_id"])

    print(f"Retrieving from LTM for {runtime.context['user_id']}")
    item = store.get(ns, LTM_KEY)
    if item is None:
        return "No data in long-term memory."
    messages = list(item.value.get("messages", []))
    if not messages:
        return "No data in long-term memory."
    return str(messages)

In [ ]:
agent = create_agent(
    tools=[save_ltm, retrieve_ltm],
    model=model,
    store=InMemoryStore(),
    checkpointer=InMemorySaver(),
    context_schema=UserContext,
    system_prompt="""You are a helpful assistant that can answer any queries.
If the user shares personal information, preferences, or explicit approvals,
save that information using the save_ltm tool.
Before answering a query when prior facts might matter, retrieve long-term
memory using retrieve_ltm and use that context in your answer.
""",
)

In [ ]:
config_siva_t1 = {"configurable": {"thread_id": "Irshad-chat-1"}}
siva = UserContext(user_id="Irshad")

response_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Irshad"}]},
    config=config_siva_t1,
    context=siva,
)
response_1

Retrieving from LTM for Irshad


{'messages': [HumanMessage(content='My name is Siva', additional_kwargs={}, response_metadata={}, id='c3313dc8-b7c3-4503-877d-34112d79ca39'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 147, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_58b0258a33', 'id': 'chatcmpl-EeDbioWIapCxEGGImc3VUKkMGw3hG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a18646-901f-7293-9ee1-98fc3f97f32a-0', tool_calls=[{'name': 'save_ltm', 'args': {'data': "User's name is Siva."}, 'id': 'call_wpfFDr05uBj5JSWmNvX2nNQ

In [ ]:
config_siva_t2 = {"configurable": {"thread_id": "Irshad-chat-2"}}

response_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_siva_t2,
    context=siva,
)
response_2

Retrieving from LTM for Irshad


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='a310b301-4082-4902-a69f-d8518d722e06'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 147, 'total_tokens': 163, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_58b0258a33', 'id': 'chatcmpl-EeDdRIJC1FpLGBrYcD71tm2KzEP1w', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a18648-31f0-72f3-82da-992414361920-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'user name'}, 'id': 'call_G1TSr2IAIt5u8EwHzZ9IrZH5', 't

In [ ]:
config_siva_t2 = {"configurable": {"thread_id": "siva-chat-2"}}

response_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_siva_t2,
    context=siva,
)
response_2

Retrieving from LTM for Irshad


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5793d088-67b4-421f-85ca-7d24ca2edaac'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_58b0258a33', 'id': 'chatcmpl-EeDddgn3UsDtdcQCWMguDGpiDOoE2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a18648-61a2-74a1-bdb5-18ffa5a6ead0-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_UX65mdzNjCWrKNJB7qciK3br', 'type':

In [ ]:
config_hrithik_t1 = {"configurable": {"thread_id": "hrithik-chat-1"}}
hrithik = UserContext(user_id="hrithik")

response_3 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_hrithik_t1,
    context=hrithik,
)
response_3

Retrieving from LTM for hrithik


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='d7192def-c7f0-4ca1-8774-14240cfbe296'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_58b0258a33', 'id': 'chatcmpl-EeDe3IIJhHqHCgvqcpsuXSlDcs3vG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a18648-c723-70f8-a35c-b694fb222bd6-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_KZRvQuAU4iykVf9gCsce95hF', 'type':